### [Class 4] Question 1. Attaching LoRA Adapters to a Base Model with PEFT

**Context:** Turning a frozen base model into a LoRA-trainable one is a three-step move: load the
pretrained model, describe the adaptation in a config (which rank, which scaling factor, which
weight matrices to adapt), and wrap the model so the adapters are injected in place. What comes
back is the same network with a pair of small trainable matrices bolted alongside each targeted
projection — the original weights untouched and frozen. Reading the injected adapters back off the
wrapped model is how you confirm the config did what you intended: the right layers were hit, and
each adapter has the rank you asked for.

**Problem Statement:**

Using the `transformers` and `peft` libraries, write a function
`lora_adapter_shapes(model_id, r, lora_alpha, target_modules)`.

1. Load the base causal language model with `AutoModelForCausalLM.from_pretrained(model_id)`.
2. Build a `LoraConfig` for a decoder-only language model (`task_type="CAUSAL_LM"`) using `r` as
   the rank, `lora_alpha` as the scaling factor, `0.05` dropout, no bias training
   (`bias="none"`), and `target_modules` as the list of module names to adapt.
3. Wrap the base model with `get_peft_model` so the adapters are injected.
4. For each adapted module, read the shapes of its **two low-rank matrices** off the wrapped
   model — the down-projection and the up-projection that were injected next to the frozen
   weights. The same set of modules is adapted inside every transformer block and the shapes are
   identical in each, so report every adapted module name exactly once.
5. Return a dictionary with exactly two keys:
   - `"scaling"` — the effective scaling factor the adapter applies, as a float
   - `"adapters"` — a dictionary mapping each adapted module's name to a list of four integers:
     the down-projection's row count, its column count, then the up-projection's row count and
     its column count

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: model_id (string)
    Single line: a string
  Argument 2: r (integer)
    Single line: one integer
  Argument 3: lora_alpha (integer)
    Single line: one integer
  Argument 4: target_modules (list_string)
    Line 1: N (list length)
    Next N lines: one string each
```

**Output Format:**

For each test case, print the result of `lora_adapter_shapes()` on a new line.

**Sample Input:**

```
model_id = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
r = 4
lora_alpha = 8
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
```

**Sample Output:**

```
{'scaling': 2.0, 'adapters': {'q_proj': [4, 8, 8, 4], 'k_proj': [4, 8, 4, 4], 'v_proj': [4, 8, 4, 4], 'o_proj': [4, 8, 8, 4], 'gate_proj': [4, 8, 32, 4], 'up_proj': [4, 8, 32, 4], 'down_proj': [4, 32, 8, 4]}}
```

**Note:** The input handling is already in place — you only need to implement the function. The
models used here are tiny test checkpoints, so loading one inside the function is cheap. Take the
shapes from the wrapped model itself rather than reasoning them out from the config — that is the
point of inspecting what was injected. Adapt only the modules named in `target_modules`: a module
the list doesn't name must not appear in the result, even if it is a linear layer. Note also that
one of the two matrices has the incoming feature count on one side and the other has the outgoing
feature count — a module whose input and output widths differ will make that visible, so don't
assume the two shapes are transposes of a square.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: lora_adapter_shapes
RETURN TYPE: Dictionary

ARGUMENTS:
1. model_id       → String
2. r              → Integer
3. lora_alpha     → Integer
4. target_modules → List of Strings
```

---

### 3. Reference Solution Code

```python
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def lora_adapter_shapes(model_id: str, r: int, lora_alpha: int, target_modules: list) -> dict:
    base_model = AutoModelForCausalLM.from_pretrained(model_id)

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        target_modules=target_modules,
    )
    peft_model = get_peft_model(base_model, lora_config)

    sample_layer = peft_model.model.model.layers[0]

    adapters = {}
    for name in target_modules:
        if "proj" in name and ("q_" in name or "k_" in name or "v_" in name or "o_" in name):
            module = getattr(sample_layer.self_attn, name)
        else:
            module = getattr(sample_layer.mlp, name)

        if hasattr(module, "lora_A"):
            A_shape = module.lora_A["default"].weight.shape   # (r, in_features)
            B_shape = module.lora_B["default"].weight.shape   # (out_features, r)
            adapters[name] = [A_shape[0], A_shape[1], B_shape[0], B_shape[1]]

    return {"scaling": lora_alpha / r, "adapters": adapters}
```

---

### 4. Test Cases

**in1.txt**
```
1
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5
4
8
7
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj
```

**out1.txt**
```
{'scaling': 2.0, 'adapters': {'q_proj': [4, 8, 8, 4], 'k_proj': [4, 8, 4, 4], 'v_proj': [4, 8, 4, 4], 'o_proj': [4, 8, 8, 4], 'gate_proj': [4, 8, 32, 4], 'up_proj': [4, 8, 32, 4], 'down_proj': [4, 32, 8, 4]}}
```

**in2.txt**
```
1
trl-internal-testing/small-Qwen2ForCausalLM-2.5
8
24
2
q_proj
v_proj
```

**out2.txt**
```
{'scaling': 3.0, 'adapters': {'q_proj': [8, 128, 128, 8], 'v_proj': [8, 128, 64, 8]}}
```

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


def lora_adapter_shapes(model_id: str, r: int, lora_alpha: int, target_modules: list) -> dict:
    base_model = AutoModelForCausalLM.from_pretrained(model_id)

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        target_modules=target_modules,
    )
    peft_model = get_peft_model(base_model, lora_config)

    sample_layer = peft_model.model.model.layers[0]

    adapters = {}
    for name in target_modules:
        if "proj" in name and ("q_" in name or "k_" in name or "v_" in name or "o_" in name):
            module = getattr(sample_layer.self_attn, name)
        else:
            module = getattr(sample_layer.mlp, name)

        if hasattr(module, "lora_A"):
            A_shape = module.lora_A["default"].weight.shape   # (r, in_features)
            B_shape = module.lora_B["default"].weight.shape   # (out_features, r)
            adapters[name] = [A_shape[0], A_shape[1], B_shape[0], B_shape[1]]

    return {"scaling": lora_alpha / r, "adapters": adapters}

In [ ]:
model_id = 'trl-internal-testing/tiny-Qwen2ForCausalLM-2.5'
r = 4
lora_alpha = 8
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

lora_adapter_shapes(model_id, r, lora_alpha, target_modules)

In [ ]:
model_id = 'trl-internal-testing/small-Qwen2ForCausalLM-2.5'
r = 8
lora_alpha = 24
target_modules = ['q_proj', 'v_proj']

lora_adapter_shapes(model_id, r, lora_alpha, target_modules)

### [Class 4] Question 2. How Much Does LoRA Shrink the Trainable-Parameter Count?

**Context:** Full fine-tuning trains every parameter, and with AdamW in mixed precision that costs
roughly 10 bytes per parameter once weights, gradients and optimizer states are counted — about
70 GB for a 7B model, beyond any single consumer GPU. LoRA sidesteps this by freezing the
pretrained weights and training only a pair of low-rank matrices alongside each adapted layer, so
the trainable set collapses to the order of one percent of the model — often far less. That fraction
is worth computing from the config *before* launching a run: it decides whether the job fits on the
GPU you have, how large the optimizer state and the saved adapter will be, and how far you can push
the rank or the number of adapted modules.

**Problem Statement:**

Write a function `lora_trainable_reduction(base_params, target_modules, r, num_layers)`.

1. `base_params` is the total parameter count of the frozen base model — every one of which full
   fine-tuning would train.
2. `target_modules` is a list of `[d_in, d_out]` pairs, one pair per weight matrix that LoRA
   adapts inside a single transformer block. The same set of modules repeats in every one of the
   model's `num_layers` blocks.
3. For each adapted matrix, the original `d_in × d_out` weights stay **frozen** and LoRA instead
   trains two rank-`r` matrices in parallel with them: a down-projection of shape `r × d_in` and
   an up-projection of shape `d_out × r`. Count the parameters those two matrices contribute. No
   bias terms are trained.
4. Sum that count over every target module in one block, then scale by `num_layers` to get the
   model-wide trainable total.
5. Return the percentage reduction in trainable parameters compared with full fine-tuning,
   measured against `base_params` — that is,
   `100 × (base_params − trainable_total) / base_params` — rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: base_params (integer)
    Single line: one integer
  Argument 2: target_modules (2D list of ints)
    Line 1: R C (number of rows, number of columns — space-separated)
    Next R lines: C space-separated integers each
      (each row is one [d_in, d_out] pair, so C is always 2)
  Argument 3: r (integer)
    Single line: one integer
  Argument 4: num_layers (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `lora_trainable_reduction()` on a new line.

**Sample Input:**

```
base_params = 1543714304
target_modules = [[1536, 1536],   # q_proj
                  [1536, 256],    # k_proj
                  [1536, 256],    # v_proj
                  [1536, 1536],   # o_proj
                  [1536, 8960],   # gate_proj
                  [1536, 8960],   # up_proj
                  [8960, 1536]]   # down_proj
r = 16
num_layers = 28
```

**Sample Output:**

```
98.8039
```

**Note:** The input handling is already in place — you only need to implement the function. Nothing
needs to be loaded or trained; this is arithmetic on a configuration. The two low-rank matrices
share `r` as their inner dimension, which is what makes the count scale with `d_in + d_out` rather
than with `d_in × d_out`. A module's two dimensions are often unequal — key/value projections can be
much narrower than the query projection, and MLP projections are wider on one side — so keep each
pair's roles straight instead of assuming square matrices.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: lora_trainable_reduction
RETURN TYPE: Float

ARGUMENTS:
1. base_params    → Integer
2. target_modules → 2D List of Ints
3. r              → Integer
4. num_layers     → Integer
```

---

### 3. Reference Solution Code

```python
def lora_trainable_reduction(base_params: int, target_modules: list[list[int]],
                             r: int, num_layers: int) -> float:
    per_layer = sum(r * (d_in + d_out) for d_in, d_out in target_modules)
    trainable_total = per_layer * num_layers
    return round(100.0 * (base_params - trainable_total) / base_params, 4)
```

---

### 4. Test Cases

**in1.txt**
```
1
1543714304
7 2
1536 1536
1536 256
1536 256
1536 1536
1536 8960
1536 8960
8960 1536
16
28
```

**out1.txt**
```
98.8039
```

**in2.txt**
```
1
6738415616
2 2
4096 4096
4096 4096
8
32
```

**out2.txt**
```
99.9378
```

In [ ]:
def lora_trainable_reduction(base_params: int, target_modules: list[list[int]],
                             r: int, num_layers: int) -> float:
    per_layer = sum(r * (d_in + d_out) for d_in, d_out in target_modules)
    trainable_total = per_layer * num_layers
    return round(100.0 * (base_params - trainable_total) / base_params, 4)

In [ ]:
base_params = 1543714304
target_modules = [
    [1536, 1536],
    [1536, 256],
    [1536, 256],
    [1536, 1536],
    [1536, 8960],
    [1536, 8960],
    [8960, 1536]
]
r = 16
num_layers = 28

lora_trainable_reduction(base_params, target_modules, r, num_layers)

In [ ]:
base_params = 6738415616
target_modules = [
    [4096, 4096],
    [4096, 4096]
]
r = 8
num_layers = 32

lora_trainable_reduction(base_params, target_modules, r, num_layers)